# Generador de clientes sintéticos de México

Este notebook crea una base de personas ficticias a partir de los CSV del directorio de entrada. No utiliza ni produce datos personales reales.

`N_CLIN` es exactamente el número de clientes sintéticos a generar: por ejemplo, `N_CLIN = 1000` produce 1,000 registros.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Semilla para resultados reproducibles. Cambiarla genera otra población sintética.
SEMILLA = 20261019
N_CLIN = 1_000

# Funciona tanto si Jupyter se abre desde la raíz del proyecto como desde esta carpeta.
RUTA_PROYECTO = next(
    (ruta for ruta in (Path.cwd(), Path.cwd().parent)
     if (ruta / 'SyntethicDataGenerator - Input').exists()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError('No se encontró SyntethicDataGenerator - Input.')
RUTA_INPUT = RUTA_PROYECTO / 'SyntethicDataGenerator - Input'
RUTA_OUTPUT = RUTA_PROYECTO / 'SyntethicDataGenerator' / 'output'
RUTA_OUTPUT.mkdir(exist_ok=True)

rng = np.random.default_rng(SEMILLA)

In [ ]:
nombres = pd.read_csv(RUTA_INPUT / 'nombres_mexico_500.csv', encoding='utf-8')
apellidos = pd.read_csv(RUTA_INPUT / 'apellidos_mexico_200.csv', encoding='utf-8')

# Controles de calidad de los insumos.
assert len(nombres) == 500, 'Se esperan 500 nombres.'
assert nombres['genero'].value_counts().to_dict() == {'Mujer': 250, 'Hombre': 250}
assert len(apellidos) == 200, 'Se esperan 200 apellidos.'

display(nombres.groupby('genero').size().rename('cantidad'))
display(apellidos.head())

In [ ]:
# Primero se selecciona el género y después un nombre compatible.
generos = rng.choice(['Mujer', 'Hombre'], size=N_CLIN)
nombres_por_genero = {
    genero: grupo['nombre'].to_numpy()
    for genero, grupo in nombres.groupby('genero')
}

nombre_elegido = np.array([
    rng.choice(nombres_por_genero[genero]) for genero in generos
])
apellidos_disponibles = apellidos['apellido'].to_numpy()

clientes = pd.DataFrame({
    'id_cliente_sintetico': np.arange(1, N_CLIN + 1),
    'nombre': nombre_elegido,
    'apellido_paterno': rng.choice(apellidos_disponibles, size=N_CLIN),
    'apellido_materno': rng.choice(apellidos_disponibles, size=N_CLIN),
    'genero': generos,
})
clientes['nombre_completo'] = (
    clientes['nombre'] + ' ' + clientes['apellido_paterno'] + ' ' + clientes['apellido_materno']
)

# Orden pensado para consumirlo en la siguiente fase del WebScraping.
clientes = clientes[[
    'id_cliente_sintetico', 'nombre', 'apellido_paterno', 'apellido_materno',
    'nombre_completo', 'genero'
]]
clientes.head(10)

In [ ]:
# Exportación UTF-8 con BOM: Excel conserva acentos y ñ al abrir el CSV.
archivo_salida = RUTA_OUTPUT / f'clientes_sinteticos_{N_CLIN}.csv'
clientes.to_csv(archivo_salida, index=False, encoding='utf-8-sig')

print(f'{len(clientes):,} clientes generados en: {archivo_salida.resolve()}')
display(clientes['genero'].value_counts().rename('cantidad'))